# 地合い(market_climate) 実験用ノートブック

`technical-practice/market_climate.py` と同じロジックを使い、Google Colab上で対話的に結果を見たり、
セクター温度感(上昇/買い集め/中立/調整)などの**自前設計した閾値**を試行錯誤するためのノートブック。

## 使い方 (Google Colab)
1. Colabで「ファイル」→「ノートブックを開く」→「GitHub」タブを選び、このリポジトリのURL
   (`https://github.com/masatomaro-bot/ai-bubble-watch`) を貼り付けて、
   `technical-practice/notebooks/market_climate_explorer.ipynb` を開く
2. 上から順にセルを実行する(1つ目のセルでリポジトリをクローンして依存パッケージを入れる)

## 注意
- ここで計算する値は `market_climate.py` 本体と同じ関数を使っているが、**日次の自動実行
  (GitHub Actions) には影響しない**。あくまで手元で試すための場所。
- セクター温度感・RISK-ON/OFF判定の閾値は完全に自前設計であり、オニミネサイト
  (oniminetrade.com) の非公開ロジックと一致する保証はない(詳細は `../README.md` 参照)。
  ここでの調整はあくまで「自分の目で見て納得できる分類か」を確認・調整するためのもの。

In [ ]:
# リポジトリをクローンして依存パッケージをインストール(初回のみ実行)
!git clone --depth 1 https://github.com/masatomaro-bot/ai-bubble-watch.git
%cd ai-bubble-watch/technical-practice
!pip install -q -r requirements.txt

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

import matplotlib.pyplot as plt
import pandas as pd

import market_climate as mc
import universe

## 1. 本日の地合い判定をまるごと確認する

`market_climate.run()` は実際の日次自動実行(GitHub Actions)と全く同じ関数。
母集団を絞りたい場合は `max_breadth_tickers` を指定する(実行時間短縮のため、まずは小さめを推奨)。

In [ ]:
result = mc.run(breadth_universe="broad", max_breadth_tickers=300)
result["market_regime"], result["breadth_universe_source"], result["breadth_extended"]

## 2. セクター/テーマのRRGマップを散布図で見る

横軸: RS-Ratio(相対力の水準)、縦軸: RS-Momentum(相対力の勢い)。
右上=主導、左上=改善、左下=遅行、右下=鈍化、という4象限で色分けする。

In [ ]:
sector_tickers = list(mc.SECTOR_ETFS.keys()) + list(mc.THEME_ETFS.keys()) + [mc.BENCHMARK_FOR_SECTORS]
hist = mc.download_history(sector_tickers, period="1y")
benchmark_close = hist[mc.BENCHMARK_FOR_SECTORS]["Close"]

all_etfs = {**mc.SECTOR_ETFS, **mc.THEME_ETFS}
rows = []
for etf, name in all_etfs.items():
    if etf not in hist or hist[etf].empty:
        continue
    close = hist[etf]["Close"]
    rrg = mc.compute_sector_rrg(close, benchmark_close)
    temperature = mc.classify_sector_temperature(close, benchmark_close) if etf in mc.SECTOR_ETFS else None
    rows.append({"etf": etf, "name_jp": name, "is_sector": etf in mc.SECTOR_ETFS, "temperature": temperature, **rrg})

rrg_df = pd.DataFrame(rows)
rrg_df

In [ ]:
quadrant_colors = {"主導": "tab:green", "改善": "tab:blue", "鈍化": "tab:orange", "遅行": "tab:red", "不明": "gray"}

fig, ax = plt.subplots(figsize=(8, 8))
for _, r in rrg_df.iterrows():
    if pd.isna(r["rs_ratio"]) or pd.isna(r["rs_momentum"]):
        continue
    marker = "o" if r["is_sector"] else "^"
    ax.scatter(r["rs_ratio"], r["rs_momentum"], color=quadrant_colors.get(r["quadrant"], "gray"), marker=marker, s=80)
    ax.annotate(r["etf"], (r["rs_ratio"], r["rs_momentum"]), fontsize=8, xytext=(4, 4), textcoords="offset points")

ax.axhline(100, color="black", lw=0.6)
ax.axvline(100, color="black", lw=0.6)
ax.set_xlabel("RS-Ratio")
ax.set_ylabel("RS-Momentum")
ax.set_title("セクター(丸)/テーマ(三角) RRGマップ")
plt.show()

## 3. セクター温度感の閾値を試しに変えてみる

`market_climate.RS_TREND_WINDOW`(相対力の変化を見る期間、既定20営業日)を変えると、
「上昇/買い集め/中立/調整」の分類やRISK-ON/OFF判定がどう変わるかを確認できる。

ipywidgetsのスライダーがColab上で動かない場合は、`rs_window`の値を直接書き換えて再実行してもよい。

In [ ]:
def reclassify_with_window(rs_window: int = 20):
    original_window = mc.RS_TREND_WINDOW
    mc.RS_TREND_WINDOW = rs_window
    try:
        temperatures = {}
        for etf in mc.SECTOR_ETFS:
            if etf not in hist or hist[etf].empty:
                continue
            temperatures[etf] = mc.classify_sector_temperature(hist[etf]["Close"], benchmark_close)
        regime = mc.majority_vote_market_regime(temperatures)
    finally:
        mc.RS_TREND_WINDOW = original_window

    print(f"RS_TREND_WINDOW = {rs_window}")
    for etf, temp in temperatures.items():
        print(f"  {etf} ({mc.SECTOR_ETFS[etf]}): {temp}")
    print("判定:", regime)


try:
    import ipywidgets as widgets

    widgets.interact(reclassify_with_window, rs_window=widgets.IntSlider(min=5, max=60, step=5, value=20))
except ImportError:
    reclassify_with_window(20)

## メモ

- ここでの調整結果を本番([market_climate.py](../market_climate.py))に反映したい場合は、
  `RS_TREND_WINDOW` などの定数をコード側で書き換えて、`tests/` のユニットテストを通してから
  コミットすること。
- 経済イベントカレンダーは未実装(README参照)。日付を手打ちで足したりしないこと。